# 🛒 Analyse E-Commerce Olist — Notebook 01 : Nettoyage & Jointures

**Dataset :** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

---

## 🎯 Questions Métier

1. **Quelles catégories de produits génèrent le plus de chiffre d'affaires ?**  
   → Identifier les top-catégories pour orienter les décisions merchandising.

2. **Comment évolue le chiffre d'affaires mensuel entre 2016 et 2018 ?**  
   → Détecter les tendances de croissance et les saisonnalités.

3. **Quel est l'impact du délai de livraison sur la satisfaction client (note) ?**  
   → Comprendre si les retards dégradent les avis.

4. **Quels États brésiliens sont les plus dynamiques en volume de commandes et en panier moyen ?**  
   → Cibler les marchés prioritaires pour l'expansion géographique.

---

## 📂 Structure du Dataset Olist

| Fichier | Description |
|---------|-------------|
| `olist_orders_dataset.csv` | Commandes (statut, dates) |
| `olist_customers_dataset.csv` | Clients (ville, état) |
| `olist_order_items_dataset.csv` | Articles commandés (prix, vendeur) |
| `olist_products_dataset.csv` | Produits (catégorie, dimensions) |
| `olist_order_payments_dataset.csv` | Paiements (mode, valeur) |
| `olist_order_reviews_dataset.csv` | Avis clients (note 1-5) |
| `product_category_name_translation.csv` | Traduction catégories PT→EN |

## 1️⃣ Import des librairies

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 20)

print('pandas :', pd.__version__)
print('numpy  :', np.__version__)
print('seaborn:', sns.__version__)

pandas : 3.0.2
numpy  : 2.4.4
seaborn: 0.13.2


## 2️⃣ Chargement des fichiers CSV

In [2]:
DATA_PATH = os.path.join('..', 'data', 'raw')

ORDER_DATE_COLS = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

df_orders = pd.read_csv(
    os.path.join(DATA_PATH, 'olist_orders_dataset.csv'),
    parse_dates=ORDER_DATE_COLS
)
df_customers  = pd.read_csv(os.path.join(DATA_PATH, 'olist_customers_dataset.csv'))
df_items      = pd.read_csv(os.path.join(DATA_PATH, 'olist_order_items_dataset.csv'),
                             parse_dates=['shipping_limit_date'])
df_products   = pd.read_csv(os.path.join(DATA_PATH, 'olist_products_dataset.csv'))
df_payments   = pd.read_csv(os.path.join(DATA_PATH, 'olist_order_payments_dataset.csv'))
df_reviews    = pd.read_csv(os.path.join(DATA_PATH, 'olist_order_reviews_dataset.csv'),
                             parse_dates=['review_creation_date', 'review_answer_timestamp'])
df_categories = pd.read_csv(os.path.join(DATA_PATH, 'product_category_name_translation.csv'))

print('✅ Tous les fichiers chargés avec succès.')

✅ Tous les fichiers chargés avec succès.


In [3]:
datasets = {
    'orders': df_orders, 'customers': df_customers, 'items': df_items,
    'products': df_products, 'payments': df_payments,
    'reviews': df_reviews, 'categories': df_categories
}
print(f"{'Dataset':<15} {'Lignes':>8} {'Colonnes':>10}")
print('-' * 36)
for name, df in datasets.items():
    print(f"{name:<15} {df.shape[0]:>8,} {df.shape[1]:>10}")

Dataset           Lignes   Colonnes
------------------------------------
orders            99,441          8
customers         99,441          5
items            148,676          7
products          32,951          9
payments         109,381          5
reviews           84,524          7
categories            17          2


## 3️⃣ Contrôle qualité : valeurs manquantes

In [4]:
for name, df in datasets.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f'\n📋 {name.upper()} — Valeurs manquantes :')
        for col, cnt in missing.items():
            pct = cnt / len(df) * 100
            print(f'   {col:<45} {cnt:>6,}  ({pct:.2f}%)')
    else:
        print(f'\n✅ {name.upper()} — Aucune valeur manquante.')


📋 ORDERS — Valeurs manquantes :
   order_delivered_customer_date                  3,021  (3.04%)

✅ CUSTOMERS — Aucune valeur manquante.

✅ ITEMS — Aucune valeur manquante.

✅ PRODUCTS — Aucune valeur manquante.

✅ PAYMENTS — Aucune valeur manquante.

📋 REVIEWS — Valeurs manquantes :
   review_comment_title                          28,143  (33.30%)
   review_comment_message                        35,980  (42.57%)

✅ CATEGORIES — Aucune valeur manquante.


### Explication des NaN dans `orders`

> **`order_delivered_customer_date`** : Cette colonne est naturellement `NaN` pour les commandes dont le statut est différent de `delivered`. Ce n'est pas une erreur de données — ces lignes ne seront PAS supprimées ici.

> **`order_approved_at`** : Quelques commandes non approuvées (paiement échoué). Exclues de l'analyse temporelle si nécessaire.

## 4️⃣ Contrôle qualité : doublons

In [5]:
checks = [
    ('orders', df_orders, 'order_id'),
    ('customers', df_customers, 'customer_id'),
    ('products', df_products, 'product_id'),
    ('reviews', df_reviews, 'review_id'),
]
for name, df, key in checks:
    dup_count = df.duplicated(subset=[key]).sum()
    status = '✅' if dup_count == 0 else '⚠️'
    print(f'{status} {name:<12} — doublons sur "{key}": {dup_count}')

✅ orders       — doublons sur "order_id": 0


✅ customers    — doublons sur "customer_id": 0
✅ products     — doublons sur "product_id": 0
✅ reviews      — doublons sur "review_id": 0


## 5️⃣ Filtrage : commandes livrées uniquement

In [6]:
print(f'Avant filtrage — df_orders.shape : {df_orders.shape}')
print('\nDistribution des statuts :')
print(df_orders['order_status'].value_counts().to_string())

Avant filtrage — df_orders.shape : (99441, 8)

Distribution des statuts :


order_status
delivered      96420
canceled        1028
shipped         1020
invoiced         297
unavailable      270
processing       210
approved         101
created           95


In [7]:
delivered_mask = df_orders['order_status'] == 'delivered'
n_removed = (~delivered_mask).sum()
n_total   = len(df_orders)
df_orders_clean = df_orders[delivered_mask].copy()

print(f'Après filtrage — df_orders_clean.shape : {df_orders_clean.shape}')
print(f'\n🗑️  Suppression de {n_removed:,} lignes non-livrées, soit {n_removed/n_total*100:.2f}% du dataset.')

Après filtrage — df_orders_clean.shape : (96420, 8)

🗑️  Suppression de 3,021 lignes non-livrées, soit 3.04% du dataset.


### Décision documentée

> **Suppression des commandes non livrées** : Seules les commandes `delivered` sont conservées. Les statuts `canceled`, `unavailable`, etc. ne représentent pas de transactions complètes et biaiseraient les analyses de CA, délais et satisfaction.

## 6️⃣ Nettoyage produits : catégories manquantes

In [8]:
print(f'Avant — df_products.shape : {df_products.shape}')
n_cat_missing = df_products['product_category_name'].isna().sum()
print(f'Produits sans catégorie : {n_cat_missing:,} ({n_cat_missing/len(df_products)*100:.2f}%)')

df_products_clean = df_products.copy()
df_products_clean['product_category_name'] = df_products_clean['product_category_name'].fillna('unknown')

print(f'Après — df_products_clean.shape : {df_products_clean.shape}')
print("\n✅ NaN de 'product_category_name' remplacés par 'unknown' (0 suppression).")

Avant — df_products.shape : (32951, 9)
Produits sans catégorie : 0 (0.00%)
Après — df_products_clean.shape : (32951, 9)

✅ NaN de 'product_category_name' remplacés par 'unknown' (0 suppression).


## 7️⃣ Jointures

In [9]:
# ── Jointure 1 : Orders + Customers
print('Jointure : orders + customers')
print(f'  avant  : {df_orders_clean.shape}')
df_orders_customers = pd.merge(df_orders_clean, df_customers, on='customer_id', how='left')
print(f'  après  : {df_orders_customers.shape}')
assert df_orders_customers.shape[0] == df_orders_clean.shape[0]
print('  ✅ Contrôle OK — nombre de lignes inchangé.')

Jointure : orders + customers
  avant  : (96420, 8)


  après  : (96420, 12)
  ✅ Contrôle OK — nombre de lignes inchangé.


In [10]:
# ── Jointure 2 : Items + Products + Traduction
print('Jointure : items + products + categories')
print(f'  avant  : {df_items.shape}')
df_items_products = pd.merge(
    df_items,
    df_products_clean[['product_id', 'product_category_name']],
    on='product_id', how='left'
)
df_items_products = pd.merge(df_items_products, df_categories, on='product_category_name', how='left')
df_items_products['category_en'] = (
    df_items_products['product_category_name_english']
    .fillna(df_items_products['product_category_name'])
)
print(f'  après  : {df_items_products.shape}')

Jointure : items + products + categories
  avant  : (148676, 7)


  après  : (148676, 10)


In [11]:
# ── Jointure 3 : Master table
items_agg = df_items_products.groupby('order_id').agg(
    total_items_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    nb_items=('order_item_id', 'count')
).reset_index()

main_category = (
    df_items_products.groupby('order_id')['category_en']
    .agg(lambda x: x.mode()[0] if len(x) > 0 else 'unknown')
    .reset_index().rename(columns={'category_en': 'main_category'})
)

print(f'Jointure finale — avant : {df_orders_customers.shape}')
df_master = pd.merge(df_orders_customers, items_agg, on='order_id', how='left')
df_master = pd.merge(df_master, main_category, on='order_id', how='left')
reviews_agg = df_reviews.groupby('order_id')['review_score'].mean().reset_index()
df_master = pd.merge(df_master, reviews_agg, on='order_id', how='left')
print(f'Jointure finale — après : {df_master.shape}')
print('✅ Master table créée.')

Jointure finale — avant : (96420, 12)


Jointure finale — après : (96420, 17)
✅ Master table créée.


## 8️⃣ Feature engineering

In [12]:
df_master['delivery_days']       = (df_master['order_delivered_customer_date'] - df_master['order_purchase_timestamp']).dt.days
df_master['delivery_delay_days'] = (df_master['order_delivered_customer_date'] - df_master['order_estimated_delivery_date']).dt.days
df_master['purchase_year']       = df_master['order_purchase_timestamp'].dt.year
df_master['purchase_month']      = df_master['order_purchase_timestamp'].dt.month
df_master['purchase_month_str']  = df_master['order_purchase_timestamp'].dt.to_period('M').astype(str)
df_master['purchase_week']       = df_master['order_purchase_timestamp'].dt.isocalendar().week.astype(int)
df_master['purchase_dow']        = df_master['order_purchase_timestamp'].dt.day_name()
df_master['revenue']             = df_master['total_items_price'] + df_master['total_freight']
print('✅ Feature engineering terminé.')

✅ Feature engineering terminé.


## 9️⃣ Suppression des aberrations

In [13]:
print(f'Avant suppression aberrations — df_master.shape : {df_master.shape}')
mask_valid = (
    df_master['delivery_days'].notna() &
    (df_master['delivery_days'] >= 0) &
    (df_master['delivery_days'] <= 180)
)
n_before = len(df_master)
df_master = df_master[mask_valid].copy()
n_removed_anom = n_before - len(df_master)
print(f'Après suppression aberrations — df_master.shape : {df_master.shape}')
print(f'\n🗑️  Suppression de {n_removed_anom:,} lignes aberrantes (délai livraison invalide), soit {n_removed_anom/n_before*100:.2f}% du dataset.')

Avant suppression aberrations — df_master.shape : (96420, 25)
Après suppression aberrations — df_master.shape : (96420, 25)

🗑️  Suppression de 0 lignes aberrantes (délai livraison invalide), soit 0.00% du dataset.


### Décision documentée

> **Suppression des délais aberrants** : Les commandes avec délai négatif (erreur de saisie) ou > 180 jours (outlier extrême) sont exclues. Ces valeurs fausseraient l'analyse satisfaction vs. livraison.

## 🔟 Sauvegarde

In [14]:
output_path = os.path.join('..', 'data', 'master_clean.csv')
df_master.to_csv(output_path, index=False)
print(f'✅ Dataset nettoyé sauvegardé : {output_path}')
print(f'   Shape finale : {df_master.shape}')

✅ Dataset nettoyé sauvegardé : ..\data\master_clean.csv
   Shape finale : (96420, 25)


In [15]:
print('=' * 55)
print('  RÉSUMÉ DU NETTOYAGE')
print('=' * 55)
print(f'  Commandes brutes           : {df_orders.shape[0]:>10,}')
print(f'  Commandes retenues         : {df_master.shape[0]:>10,}')
print(f'  Colonnes dans master table : {df_master.shape[1]:>10}')
print(f'  Période couverte           : {df_master["order_purchase_timestamp"].min().date()} → {df_master["order_purchase_timestamp"].max().date()}')
print(f'  CA total estimé            : R$ {df_master["revenue"].sum():>12,.2f}')
print(f'  Panier moyen               : R$ {df_master["revenue"].mean():>12,.2f}')
print('=' * 55)

  RÉSUMÉ DU NETTOYAGE
  Commandes brutes           :     99,441
  Commandes retenues         :     96,420
  Colonnes dans master table :         25
  Période couverte           : 2016-10-01 → 2018-08-31
  CA total estimé            : R$ 16,987,710.68
  Panier moyen               : R$       176.18
